In [1]:
import os
import pandas as pd
os.listdir("/kaggle/input/models/devinshende/mbart50lora/pytorch/default/1/akkadian-finetuned-lora-final")

['adapter_model.safetensors',
 'training_args.bin',
 'adapter_config.json',
 'README.md',
 'tokenizer.json',
 'tokenizer_config.json',
 'special_tokens_map.json',
 'sentencepiece.bpe.model']

In [2]:
import torch
import kagglehub
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from peft import PeftModel

# Download base model from Kaggle Hub
base_root = kagglehub.model_download("kylejameshanger/akkadian-mbart-epoch5/pyTorch/default")
base_model_path = f"{base_root}/mbart-deep-past/checkpoint-1755"


# Your LoRA adapter folder
adapter_path = "/kaggle/input/models/devinshende/mbart50lora/pytorch/default/1/akkadian-finetuned-lora-final"


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = MBart50TokenizerFast.from_pretrained(adapter_path, local_files_only=True)
base_model = MBartForConditionalGeneration.from_pretrained(base_model_path, local_files_only=True)
model = PeftModel.from_pretrained(base_model, adapter_path).to(device).eval()

SRC_LANG = "ar_AR"
TGT_LANG = "en_XX"
tokenizer.src_lang = SRC_LANG

def translate(text, max_length=256):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=4,
            forced_bos_token_id=tokenizer.lang_code_to_id[TGT_LANG],
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(translate("ṭup-pu-um ša 10 ma-na KÙ.BABBAR"))

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Tablet concerning 10 minas of silver.


In [4]:
# Load test data
path = '/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv'
test_df = pd.read_csv(path)

MAX_LENGTH = 512
# Generate predictions
predictions = []
for akkadian_text in test_df['transliteration']:
    inputs = tokenizer(akkadian_text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True).to(device)
    with torch.no_grad():
        generated = model.generate(**inputs, max_length=MAX_LENGTH, num_beams=4, forced_bos_token_id=tokenizer.lang_code_to_id[TGT_LANG])
    predictions.append(tokenizer.decode(generated[0], skip_special_tokens=True))

# Save results
results = pd.DataFrame({'id': test_df['id'], 'prediction': predictions})
results.to_csv('submission.csv', index=False)
print(f"Predictions saved to test_predictions.csv ({len(results)} rows)")


Predictions saved to test_predictions.csv (4 rows)
